In [1]:
# import sys
# !{sys.executable} -m pip install opentrons
#!pip install opentrons

In [2]:
## Imports
from opentrons import protocol_api, execute, simulate
from opentrons.types import Location, Point
import pandas as pd
import numpy as np
#from data import data_colordf

# Call protocol to simulate or Execute

In [3]:
protocol = simulate.get_protocol_api('2.15')
#protocol = execute.get_protocol_api('2.15')

/Users/clarat/.opentrons/robot_settings.json not found. Loading defaults
Deck calibration not found.
/Users/clarat/.opentrons/deck_calibration.json not found. Loading defaults


# Define labwear

In [4]:
## pipette and labware
tiprack3 = protocol.load_labware('opentrons_96_tiprack_300ul',location='8')
tiprack10 = protocol.load_labware('opentrons_96_tiprack_1000ul',location='7')
pip3 = protocol.load_instrument('p300_single_gen2', 'right', tip_racks=[tiprack3])
pip10 = protocol.load_instrument('p1000_single_gen2', 'left', tip_racks=[tiprack10])
#plate96 = protocol.load_labware('corning_96_wellplate_360ul_flat',location='2')
stock = protocol.load_labware('corning_6_wellplate_16.8ml_flat',location='4')
mother = protocol.load_labware('corning_6_wellplate_16.8ml_flat',location='5')
hs_mod = protocol.load_module(module_name="heaterShakerModuleV1", location="3")
hs_adapter = hs_mod.load_adapter("opentrons_96_flat_bottom_adapter")
hs_plate = hs_adapter.load_labware("nest_96_wellplate_200ul_flat")


# Relevant Functions

In [5]:
# funtion for moving liquid since repetative
def pipette_action(protocol, pipette, stock_volume_to_pull, stock_position_to_pull, destination_well,stockname):
    pipette.aspirate(stock_volume_to_pull, stock_position_to_pull)
    pipette.dispense(stock_volume_to_pull, destination_well)
    pipette.touch_tip()  
    #pipette.blow_out(destination_well)

In [6]:
def stock_in_all(pip,stock,stock_location,tip_max_vol,dis_tray,name):
    '''
    Args- 
    stock: 1D array of all the stock volumes
    stock_location: 1D array of the stock locations corresponding to volumes
    tip_max_vol: float num of the max volume tip can hold
    dis_tray: 1D array of all the locaitons for dispensing
    name: stock name str

    Function-
    Determine the number of times needed to get stock and dispence. 
    If the volume is larger than the tip can hold, automatically determined how many times it has to aspirate and dispence. 
    Care since stock, stock_location, dis_tray are all arrays
    '''
    for i in range(len(stock)):
        stock_amount = stock[i]
        if stock_amount == 0:
            pass
        elif stock[i] >= tip_max_vol: # eg:300 is max tip volume
            n = stock[i]//tip_max_vol
            if n > 0: 
                for count in range(int(n)):
                    pipette_action(protocol, 
                            pipette = pip, 
                            stock_volume_to_pull = tip_max_vol, 
                            stock_position_to_pull = stock_location[i], 
                            destination_well = dis_tray[i],
                            stockname= name )
                    stock_amount -= tip_max_vol
                    
                    if stock_amount <= stock[i] - tip_max_vol*n and stock_amount != 0:
                        pipette_action(protocol, 
                                    pipette = pip, 
                                    stock_volume_to_pull = (stock[i] - tip_max_vol*n),
                                    stock_position_to_pull = stock_location[i], 
                                    destination_well = dis_tray[i],
                                    stockname= name )
        else: 
            pipette_action(protocol, 
                           pipette = pip, 
                           stock_volume_to_pull = stock[i], 
                           stock_position_to_pull = stock_location[i], 
                           destination_well = dis_tray[i],
                           stockname= name )
        

In [7]:
def loc_stock(n_samples,fill):
    array = []
    #fill array
    for i in range(n_samples): 
        array.append(fill) 
    return array

def loc_mixing(start,n_sample,plate): 
    array = []
    for i in range(n_sample):
        array.append(plate.wells()[i+start].center())
    return array

#let = 'D'
#RGB_stocksandlocations = {'mix_loc': [plate[str(let)+'1'].center(), plate[str(let)+'2'].center(),plate[str(let)+'3'].center(),plate[str(let)+'4'].center(),plate[str(let)+'5'].center(),plate[str(let)+'6'].center(),plate[str(let)+'7'].center(),plate[str(let)+'8'].center(),plate[str(let)+'9'].center(),plate[str(let)+'10'].center(),plate[str(let)+'11'].center(),plate[str(let)+'12'].center()], 

# Volumes and locations

In [8]:
start_stock = {
    'water_loc': [stock.wells()[0].center(),stock.wells()[1].center()],#stock.wells()[0].center()
    #'water_loc': loc_stock(3,stock.wells()[0].center())+loc_stock(3,stock.wells()[1].center()),#stock.wells()[0].center()
    'water_vol':[20000,20000] ,# adjust the volumes for each(6) concentration
    'dmf_loc':[stock.wells()[2].center(),stock.wells()[3].center()], #stock.wells()[1].center()
    #'dmf_loc':loc_stock(3,stock.wells()[2].center())+loc_stock(3,stock.wells()[3].center()), #stock.wells()[1].center()
    'dmf_vol':[20000,20000],# adjust the volumes for each(6) concentration
    'ethanol_loc':[stock.wells()[4].center(),stock.wells()[5].center()],#stock.wells()[2].center()
    #'ethanol_loc':loc_stock(3,stock.wells()[4].center())+loc_stock(3,stock.wells()[5].center()),#stock.wells()[2].center()
    'ethanol_vol':[20000,20000]# adjust the volumes for each(6) concentration
}



start_stock = pd.DataFrame(data=start_stock)
start_stock

,water_loc,water_vol,dmf_loc,dmf_vol,ethanol_loc,ethanol_vol
0,"Location(point=Point(x=24.76, y=152.78, z=11.5...",20000,"Location(point=Point(x=63.88, y=152.78, z=11.5...",20000,"Location(point=Point(x=103.0, y=152.78, z=11.5...",20000
1,"Location(point=Point(x=24.76, y=113.66, z=11.5...",20000,"Location(point=Point(x=63.88, y=113.66, z=11.5...",20000,"Location(point=Point(x=103.0, y=113.66, z=11.5...",20000


In [9]:
loc_mixing(0,6,mother)

[Location(point=Point(x=157.26, y=152.78, z=11.57), labware=A1 of Corning 6 Well Plate 16.8 mL Flat on slot 5),
 Location(point=Point(x=157.26, y=113.66, z=11.57), labware=B1 of Corning 6 Well Plate 16.8 mL Flat on slot 5),
 Location(point=Point(x=196.38, y=152.78, z=11.57), labware=A2 of Corning 6 Well Plate 16.8 mL Flat on slot 5),
 Location(point=Point(x=196.38, y=113.66, z=11.57), labware=B2 of Corning 6 Well Plate 16.8 mL Flat on slot 5),
 Location(point=Point(x=235.5, y=152.78, z=11.57), labware=A3 of Corning 6 Well Plate 16.8 mL Flat on slot 5),
 Location(point=Point(x=235.5, y=113.66, z=11.57), labware=B3 of Corning 6 Well Plate 16.8 mL Flat on slot 5)]

In [10]:
def sum_constraint(n_samples,sum_constraint):
    for i in range(len(n_samples)):
        if sum(n_samples[i]) == 0:
          n_samples[i, :] = sum_constraint/len(n_samples[i])
        current_sum = np.sum(n_samples[i, :])
        n_samples[i, :] *= sum_constraint // current_sum
    return n_samples

In [11]:
ratio = np.array([[1, 1, 1],[2, 1, 1],[2, 1, 2],[1, 1, 2],[3,1,1],[1,2,1]])
transfer_vol = sum_constraint(ratio,10000)


In [12]:
##Calling the function

stocksandlocations = {
'water_tranf':transfer_vol.T[0] ,# adjust the volumes for each(6) concentration
'dmf_tranf':transfer_vol.T[1],# adjust the volumes for each(6) concentration
'ethanol_tranf':transfer_vol.T[2],# adjust the volumes for each(6) concentration
'mother_loc': loc_mixing(0,6,mother),
}

mother_stocks = pd.DataFrame(data=stocksandlocations)
mother_stocks

,water_tranf,dmf_tranf,ethanol_tranf,mother_loc
0,3333,3333,3333,"Location(point=Point(x=157.26, y=152.78, z=11...."
1,5000,2500,2500,"Location(point=Point(x=157.26, y=113.66, z=11...."
2,4000,2000,4000,"Location(point=Point(x=196.38, y=152.78, z=11...."
3,2500,2500,5000,"Location(point=Point(x=196.38, y=113.66, z=11...."
4,6000,2000,2000,"Location(point=Point(x=235.5, y=152.78, z=11.5..."
5,2500,5000,2500,"Location(point=Point(x=235.5, y=113.66, z=11.5..."


In [17]:
water_vol   = pd.Series.to_list(mother_stocks['water_tranf'])
dmf_vol     = pd.Series.to_list(mother_stocks['dmf_tranf'])
ethanol_vol = pd.Series.to_list(mother_stocks['ethanol_tranf'])

mother_solu_loc = pd.Series.to_list(mother_stocks['mother_loc']) #dispense

plate_tranf = np.ones([16])*200
plate_tranf.tolist()

[200.0,
 200.0,
 200.0,
 200.0,
 200.0,
 200.0,
 200.0,
 200.0,
 200.0,
 200.0,
 200.0,
 200.0,
 200.0,
 200.0,
 200.0,
 200.0]

# Run the protocol

In [14]:
## commands
protocol.home()
##--make all the mother solution first--##
#pip10.pick_up_tip()
pip10.transfer(water_vol,stock.wells()[0].center(),mother_solu_loc,blow_out=False,blowout_location="destination well" ,new_tip='once')
pip10.transfer(dmf_vol,stock.wells()[2].center(),mother_solu_loc,blow_out=False,blowout_location="destination well" ,new_tip='once')
pip10.transfer(ethanol_vol,stock.wells()[4].center(),mother_solu_loc,blow_out=True,blowout_location="destination well",mix_after=(5, 750),new_tip='once') 

##--distribute to wells--##
#pip3.transfer(250,mother_solu_loc,loc_mixing(0,96,hs_plate))


   
for line in protocol.commands(): 
    print(line)

Transferring [3333.0, 5000.0, 4000.0, 2500.0, 6000.0, 2500.0] from A1 of Corning 6 Well Plate 16.8 mL Flat on slot 4 to A1 of Corning 6 Well Plate 16.8 mL Flat on slot 5
Picking up tip from A1 of Opentrons OT-2 96 Tip Rack 1000 µL on slot 7
Aspirating 1000.0 uL from A1 of Corning 6 Well Plate 16.8 mL Flat on slot 4 at 274.7 uL/sec
Dispensing 1000.0 uL into A1 of Corning 6 Well Plate 16.8 mL Flat on slot 5 at 274.7 uL/sec
Aspirating 1000.0 uL from A1 of Corning 6 Well Plate 16.8 mL Flat on slot 4 at 274.7 uL/sec
Dispensing 1000.0 uL into A1 of Corning 6 Well Plate 16.8 mL Flat on slot 5 at 274.7 uL/sec
Aspirating 666.5 uL from A1 of Corning 6 Well Plate 16.8 mL Flat on slot 4 at 274.7 uL/sec
Dispensing 666.5 uL into A1 of Corning 6 Well Plate 16.8 mL Flat on slot 5 at 274.7 uL/sec
Aspirating 666.5 uL from A1 of Corning 6 Well Plate 16.8 mL Flat on slot 4 at 274.7 uL/sec
Dispensing 666.5 uL into A1 of Corning 6 Well Plate 16.8 mL Flat on slot 5 at 274.7 uL/sec
Aspirating 1000.0 uL from A

In [35]:
mother_solu_loc[0]
len(loc_mixing(0,16,hs_plate))

16

In [38]:
loc_mixing(16*5,16,hs_plate)

[Location(point=Point(x=369.405, y=73.055, z=78.37500000000001), labware=A11 of NEST 96 Well Plate 200 µL Flat on Opentrons 96 Flat Bottom Heater-Shaker Adapter on Heater-Shaker Module GEN1 on slot 3),
 Location(point=Point(x=369.405, y=64.055, z=78.37500000000001), labware=B11 of NEST 96 Well Plate 200 µL Flat on Opentrons 96 Flat Bottom Heater-Shaker Adapter on Heater-Shaker Module GEN1 on slot 3),
 Location(point=Point(x=369.405, y=55.055, z=78.37500000000001), labware=C11 of NEST 96 Well Plate 200 µL Flat on Opentrons 96 Flat Bottom Heater-Shaker Adapter on Heater-Shaker Module GEN1 on slot 3),
 Location(point=Point(x=369.405, y=46.055, z=78.37500000000001), labware=D11 of NEST 96 Well Plate 200 µL Flat on Opentrons 96 Flat Bottom Heater-Shaker Adapter on Heater-Shaker Module GEN1 on slot 3),
 Location(point=Point(x=369.405, y=37.055, z=78.37500000000001), labware=E11 of NEST 96 Well Plate 200 µL Flat on Opentrons 96 Flat Bottom Heater-Shaker Adapter on Heater-Shaker Module GEN1 on

In [41]:
## commands for transfering to wells
hs_mod.close_labware_latch()
pip3.transfer(plate_tranf.tolist(),mother_solu_loc[0],loc_mixing(0,16,hs_plate),blow_out=False,blowout_location="destination well" ,new_tip='once')
pip3.transfer(plate_tranf.tolist(),mother_solu_loc[1],loc_mixing(16*1,16,hs_plate),blow_out=False,blowout_location="destination well" ,new_tip='once')
pip3.transfer(plate_tranf.tolist(),mother_solu_loc[2],loc_mixing(16*2,16,hs_plate),blow_out=False,blowout_location="destination well" ,new_tip='once')
pip3.transfer(plate_tranf.tolist(),mother_solu_loc[3],loc_mixing(16*3,16,hs_plate),blow_out=False,blowout_location="destination well" ,new_tip='once')
pip3.transfer(plate_tranf.tolist(),mother_solu_loc[4],loc_mixing(16*4,16,hs_plate),blow_out=False,blowout_location="destination well" ,new_tip='once')
pip3.transfer(plate_tranf.tolist(),mother_solu_loc[5],loc_mixing(16*5,16,hs_plate),blow_out=False,blowout_location="destination well" ,new_tip='once')

hs_mod.set_and_wait_for_temperature(75)
protocol.delay(minutes=1440)
hs_mod.deactivate_heater()

for line in protocol.commands(): 
    print(line)

Transferring [3333.0, 5000.0, 4000.0, 2500.0, 6000.0, 2500.0] from A1 of Corning 6 Well Plate 16.8 mL Flat on slot 4 to A1 of Corning 6 Well Plate 16.8 mL Flat on slot 5
Picking up tip from A1 of Opentrons OT-2 96 Tip Rack 1000 µL on slot 7
Aspirating 1000.0 uL from A1 of Corning 6 Well Plate 16.8 mL Flat on slot 4 at 274.7 uL/sec
Dispensing 1000.0 uL into A1 of Corning 6 Well Plate 16.8 mL Flat on slot 5 at 274.7 uL/sec
Aspirating 1000.0 uL from A1 of Corning 6 Well Plate 16.8 mL Flat on slot 4 at 274.7 uL/sec
Dispensing 1000.0 uL into A1 of Corning 6 Well Plate 16.8 mL Flat on slot 5 at 274.7 uL/sec
Aspirating 666.5 uL from A1 of Corning 6 Well Plate 16.8 mL Flat on slot 4 at 274.7 uL/sec
Dispensing 666.5 uL into A1 of Corning 6 Well Plate 16.8 mL Flat on slot 5 at 274.7 uL/sec
Aspirating 666.5 uL from A1 of Corning 6 Well Plate 16.8 mL Flat on slot 4 at 274.7 uL/sec
Dispensing 666.5 uL into A1 of Corning 6 Well Plate 16.8 mL Flat on slot 5 at 274.7 uL/sec
Aspirating 1000.0 uL from A